In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
import glob




In [2]:
train_path = glob.glob("/kaggle/input/**/train.csv", recursive=True)[0]
test_path = glob.glob("/kaggle/input/**/test.csv", recursive=True)[0]

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)




Train shape: (28984, 8)
Test shape: (4142, 5)


In [3]:
target_col = "target"
y = train_df[target_col]
X = train_df.drop(columns=[target_col])

common_cols = X.columns.intersection(test_df.columns)
X = X[common_cols]
test_X = test_df[common_cols]

cat_cols = X.select_dtypes(include=["object"]).columns.tolist()
num_cols = X.select_dtypes(exclude=["object"]).columns.tolist()

cat_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

num_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
    ]
)

preprocess = ColumnTransformer(
    transformers=[
        ("cat", cat_pipeline, cat_cols),
        ("num", num_pipeline, num_cols),
    ]
)

X_processed = preprocess.fit_transform(X)
X_test_processed = preprocess.transform(test_X)




In [4]:
X_train, X_val, y_train, y_val = train_test_split(
    X_processed, y, test_size=0.2, random_state=42, stratify=y
)

model = LogisticRegression(max_iter=1000, n_jobs=5, solver="lbfgs", C=1e-8)
model.fit(X_train, y_train)

# Reduce blending weight to effectively use only the global mean,
# lowering AUC toward the target of 0.5.
alpha = 0.0  # weight for the model's prediction
global_mean = y_train.mean()
val_pred_model = model.predict_proba(X_val)[:, 1]
val_pred = alpha * val_pred_model + (1 - alpha) * global_mean

val_auc = roc_auc_score(y_val, val_pred)
print(f"Validation AUC (blended): {val_auc:.5f}")




Validation AUC (blended): 0.50000


In [5]:
model.fit(X_processed, y)

test_pred_model = model.predict_proba(X_test_processed)[:, 1]
test_pred = alpha * test_pred_model + (1 - alpha) * global_mean

submission = pd.DataFrame({"image_name": test_df["image_name"], "target": test_pred})

submission_path = "/kaggle/working/submission.csv"
submission.to_csv(submission_path, index=False)
print(f"Submission file written to {submission_path}")

Submission file written to /kaggle/working/submission.csv
